# Jour 2 - Feature Engineering

Pipeline autonome de création de 9 nouvelles variables, d'encodage, de split, de scaling sans data leakage et de sauvegarde des artefacts.

In [12]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [13]:
DATA_PATH = '../data/processed/train_clean.csv'
df = pd.read_csv(DATA_PATH)
print(f'Shape initiale : {df.shape}')
assert df.shape == (1458, 81), f'Shape inattendue : {df.shape}'


Shape initiale : (1458, 81)


In [14]:
new_features = [
    'TotalSF', 'TotalBath', 'Age', 'RemodAge', 'TotalPorchSF',
    'HasGarage', 'HasPool', 'HasFireplace', 'HasBsmt'
]

df['TotalSF'] = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['TotalBath'] = (
    df['FullBath']
    + 0.5 * df['HalfBath']
    + df['BsmtFullBath']
    + 0.5 * df['BsmtHalfBath']
)
df['Age'] = (df['YrSold'] - df['YearBuilt']).clip(lower=0)
df['RemodAge'] = (df['YrSold'] - df['YearRemodAdd']).clip(lower=0)
df['TotalPorchSF'] = (
    df['WoodDeckSF']
    + df['OpenPorchSF']
    + df['EnclosedPorch']
    + df['3SsnPorch']
    + df['ScreenPorch']
)
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasPool'] = (df['PoolArea'] > 0).astype(int)
df['HasFireplace'] = (df['Fireplaces'] > 0).astype(int)
df['HasBsmt'] = (df['TotalBsmtSF'] > 0).astype(int)

print('Features créées :', ', '.join(new_features))

Features créées : TotalSF, TotalBath, Age, RemodAge, TotalPorchSF, HasGarage, HasPool, HasFireplace, HasBsmt


In [15]:
missing_new_features = df[new_features].isna().sum()
print('Valeurs manquantes avant imputation :')
print(missing_new_features[missing_new_features > 0])

df[new_features] = df[new_features].fillna(df[new_features].median())

print(df[new_features].describe().T)
print('Valeurs manquantes après imputation :')
print(df[new_features].isna().sum())
print('Âges négatifs :', (df[['Age', 'RemodAge']] < 0).sum().sum())

feature_correlations = (
    df[new_features + ['SalePrice']]
    .corr()['SalePrice']
    .drop('SalePrice')
    .sort_values(ascending=False)
)
print('Corrélations avec SalePrice :')
print(feature_correlations)

assert df[new_features].isna().sum().sum() == 0
assert np.isfinite(df[new_features].to_numpy()).all()

Valeurs manquantes avant imputation :
Series([], dtype: int64)
               count         mean         std    min     25%     50%      75%  \
TotalSF       1458.0  2557.150206  774.109803  334.0  2008.5  2473.0  3002.25   
TotalBath     1458.0     2.207476    0.781341    1.0     2.0     2.0     2.50   
Age           1458.0    36.598080   30.240565    0.0     8.0    35.0    54.00   
RemodAge      1458.0    22.982167   20.636501    0.0     4.0    14.0    41.00   
TotalPorchSF  1458.0   180.810014  156.120838    0.0    45.0   164.0   265.00   
HasGarage     1458.0     0.944444    0.229140    0.0     1.0     1.0     1.00   
HasPool       1458.0     0.004115    0.064040    0.0     0.0     0.0     0.00   
HasFireplace  1458.0     0.526749    0.499455    0.0     0.0     1.0     1.00   
HasBsmt       1458.0     0.974623    0.157322    0.0     1.0     1.0     1.00   

                 max  
TotalSF       6872.0  
TotalBath        6.0  
Age            136.0  
RemodAge        60.0  
TotalPorchS

In [16]:
df_encoded = df.copy()

ordinal_quality = {'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}
ordinal_exposure = {'None': 0, 'No': 1, 'Mn': 2, 'Av': 3, 'Gd': 4}
ordinal_fin_type = {'None': 0, 'Unf': 1, 'LwQ': 2, 'Rec': 3, 'BLQ': 4, 'ALQ': 5, 'GLQ': 6}
ordinal_finish = {'None': 0, 'Unf': 1, 'RFn': 2, 'Fin': 3}
ordinal_functional = {'Sal': 0, 'Sev': 1, 'Maj2': 2, 'Maj1': 3, 'Mod': 4, 'Min2': 5, 'Min1': 6, 'Typ': 7}
ordinal_fence = {'None': 0, 'MnWw': 1, 'GdWo': 2, 'MnPrv': 3, 'GdPrv': 4}
ordinal_general = {
    'None': 0, 'N': 0, 'No': 0, 'Y': 1, 'Yes': 1,
    'ELO': 0, 'NoSeWa': 1, 'NoSewr': 2, 'AllPub': 3,
    'Sev': 0, 'Mod': 1, 'Gtl': 2,
    'IR3': 0, 'IR2': 1, 'IR1': 2, 'Reg': 3,
    'Grvl': 0, 'Pave': 1
}

mappings = {
    'ExterQual': ordinal_quality, 'ExterCond': ordinal_quality,
    'BsmtQual': ordinal_quality, 'BsmtCond': ordinal_quality,
    'HeatingQC': ordinal_quality, 'KitchenQual': ordinal_quality,
    'FireplaceQu': ordinal_quality, 'GarageQual': ordinal_quality,
    'GarageCond': ordinal_quality, 'PoolQC': ordinal_quality,
    'BsmtExposure': ordinal_exposure,
    'BsmtFinType1': ordinal_fin_type, 'BsmtFinType2': ordinal_fin_type,
    'GarageFinish': ordinal_finish,
    'Functional': ordinal_functional, 'Fence': ordinal_fence,
    'CentralAir': ordinal_general, 'PavedDrive': ordinal_general,
    'Utilities': ordinal_general, 'LandSlope': ordinal_general,
    'LotShape': ordinal_general, 'Street': ordinal_general,
    'Alley': ordinal_general
}

structural_categorical = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'MasVnrType', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtExposure', 'BsmtQual', 'BsmtCond', 'BsmtFinType1', 'BsmtFinType2'
]
for column in structural_categorical:
    if column in df_encoded.columns:
        df_encoded[column] = df_encoded[column].fillna('None')

for column, mapping in mappings.items():
    if column in df_encoded.columns:
        df_encoded[column] = df_encoded[column].map(mapping).fillna(0).astype(int)

categorical_columns = df_encoded.select_dtypes(include=['object', 'str']).columns.tolist()
df_encoded = pd.get_dummies(
    df_encoded,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

numeric_columns = df_encoded.select_dtypes(include=[np.number]).columns
df_encoded[numeric_columns] = df_encoded[numeric_columns].fillna(
    df_encoded[numeric_columns].median()
)

print(f'Shape après encodage : {df_encoded.shape}')
print('Toutes les colonnes sont numériques :', df_encoded.select_dtypes(include=['object', 'str']).empty)
assert df_encoded.shape == (1458, 211), f'Shape encodée inattendue : {df_encoded.shape}'


Shape après encodage : (1458, 211)
Toutes les colonnes sont numériques : True


In [17]:
y = df_encoded['SalePrice'].reset_index(drop=True)
X = df_encoded.drop(columns=['Id', 'SalePrice']).reset_index(drop=True)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('X_train :', X_train.shape)
print('X_val   :', X_val.shape)
print('y_train :', y_train.shape)
print('y_val   :', y_val.shape)
assert X_train.shape == (1166, 209)
assert X_val.shape == (292, 209)
assert y_train.shape == (1166,)
assert y_val.shape == (292,)


X_train : (1166, 209)
X_val   : (292, 209)
y_train : (1166,)
y_val   : (292,)


In [18]:
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_val_scaled = pd.DataFrame(
    scaler.transform(X_val),
    columns=X_val.columns,
    index=X_val.index
)

train_non_constant = X_train.columns[X_train.std(ddof=0) > 0]
print('Moyenne absolue maximale train :', X_train_scaled.mean().abs().max())
print('Ecart-type moyen des colonnes variables :', X_train_scaled[train_non_constant].std(ddof=0).mean())
assert X_train_scaled.mean().abs().max() < 1e-10
assert np.allclose(
    X_train_scaled[train_non_constant].std(ddof=0).to_numpy(),
    1.0,
    atol=1e-10
)

Moyenne absolue maximale train : 3.408213296110009e-14
Ecart-type moyen des colonnes variables : 1.0


In [19]:
joblib.dump(X_train_scaled, '../data/processed/X_train.pkl')
joblib.dump(X_val_scaled, '../data/processed/X_val.pkl')
joblib.dump(y_train, '../data/processed/y_train.pkl')
joblib.dump(y_val, '../data/processed/y_val.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('Nouveaux .pkl avec 209 features sauvegardés')

Nouveaux .pkl avec 209 features sauvegardés


In [20]:
X_train_check = joblib.load('../data/processed/X_train.pkl')
X_val_check = joblib.load('../data/processed/X_val.pkl')
y_train_check = joblib.load('../data/processed/y_train.pkl')
y_val_check = joblib.load('../data/processed/y_val.pkl')
scaler_check = joblib.load('../models/scaler.pkl')

print('X_train :', X_train_check.shape)
print('X_val   :', X_val_check.shape)
print('y_train :', y_train_check.shape)
print('y_val   :', y_val_check.shape)
print('scaler  :', scaler_check.n_features_in_, 'features')
print('Colonnes ajoutées :', ', '.join(new_features))

assert X_train_check.shape == (1166, 209)
assert X_val_check.shape == (292, 209)
assert y_train_check.shape == (1166,)
assert y_val_check.shape == (292,)
assert scaler_check.n_features_in_ == 209
assert all(column in X_train_check.columns for column in new_features)


X_train : (1166, 209)
X_val   : (292, 209)
y_train : (1166,)
y_val   : (292,)
scaler  : 209 features
Colonnes ajoutées : TotalSF, TotalBath, Age, RemodAge, TotalPorchSF, HasGarage, HasPool, HasFireplace, HasBsmt
